# 04 — Live Demo

**VisionMind — ITAI 1378 Final Project**  
**Author:** Ahmet Burak Solak

End-user demo: drop in any natural image (your phone, the web, the CIFAR-10 test set, …) and the system will return:

1. The top-1 prediction from each of the three models
2. The **ensemble** prediction
3. A **Grad-CAM** explanation overlay

This is the notebook that backs the recorded demo video.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
if '..' not in sys.path:
    sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.inference import load_models, predict
from src.explainability import save_gradcam_overlay

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 1. Load the three trained models

In [ ]:
checkpoints = {
    'custom_cnn'  : '../models/custom_cnn_best.pt',
    'resnet18'    : '../models/resnet18_best.pt',
    'mobilenet_v2': '../models/mobilenet_v2_best.pt',
}
models_ = load_models(checkpoints, device=DEVICE)
print('Loaded:', list(models_.keys()))

## 2. Pick or upload an image

By default this notebook uses the first image in `data/sample/` (committed to the repo). Replace `IMAGE_PATH` with any path on your machine.

In [ ]:
sample_dir = Path('../data/sample')
candidates = sorted(sample_dir.glob('*.png')) + sorted(sample_dir.glob('*.jpg'))
if not candidates:
    raise FileNotFoundError('No sample images found. Add a PNG/JPG under data/sample/')
IMAGE_PATH = candidates[0]
print('Using image:', IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert('RGB')
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis('off')
plt.title('Input')
plt.show()

## 3. Predict with each model + ensemble

In [ ]:
result = predict(image, models_, device=DEVICE)
for name, info in result.items():
    print(f'{name:>14s}: {info["class"]:<10s} ({info["confidence"]*100:5.2f}%)')

### Per-class probabilities (ensemble)

In [ ]:
if 'ensemble' in result:
    probs = result['ensemble']['probs']
    items = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
    classes, values = zip(*items)
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.bar(classes, values, color='#22c55e')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Ensemble probability')
    ax.set_title('VisionMind — ensemble class probabilities')
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2f}',
                ha='center', fontsize=8)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 4. Grad-CAM explanation

We use the strongest single model (ResNet18 if available) for the explanation overlay.

In [ ]:
explain_name = 'resnet18' if 'resnet18' in models_ else next(iter(models_.keys()))
out_path = Path(f'../results/images/demo_gradcam_{IMAGE_PATH.stem}.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
label, conf = save_gradcam_overlay(
    image=image,
    model=models_[explain_name],
    model_name=explain_name,
    out_path=out_path,
    device=DEVICE,
)
print(f'Grad-CAM saved to {out_path} | predicted={label} ({conf*100:.1f}%)')

from IPython.display import Image as IPyImage
IPyImage(filename=str(out_path))

## 5. Try your own image

Replace `MY_IMAGE` with any local path. Larger images (224×224 or up) work best for transfer-learning models — the demo automatically resizes.

In [ ]:
# MY_IMAGE = '/path/to/your/image.jpg'
# my_image = Image.open(MY_IMAGE).convert('RGB')
# print(predict(my_image, models_, device=DEVICE))
# save_gradcam_overlay(my_image, models_[explain_name], explain_name, 'my_gradcam.png', DEVICE)
print('(Uncomment the cell above and point it at your image.)')